# Disease Prediction using Machine Learning

**Dataset:** Breast Cancer Wisconsin Diagnostic dataset (built into scikit-learn)

**Goal:** Train and compare Logistic Regression, Random Forest, and Support Vector Machine classifiers.

> Educational demonstration only. This model is not a medical device and must not be used to diagnose or guide treatment.

## 1. Imports and dataset loading

The target encoding is provided by scikit-learn. The target names are `malignant` and `benign`; target value `1` is `benign`.

In [ ]:
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score, roc_auc_score,
    ConfusionMatrixDisplay, RocCurveDisplay
)

RANDOM_STATE = 42
RESULTS_DIR = Path("results")
RESULTS_DIR.mkdir(exist_ok=True)

dataset = load_breast_cancer(as_frame=True)
X = dataset.data
y = dataset.target

print("Feature matrix:", X.shape)
print("Target names:", dict(enumerate(dataset.target_names)))
print("Missing values:", int(X.isna().sum().sum()))
display(X.head())

## 2. Preprocessing and train/test split

The dataset is split into training and test sets using stratification. Standardization is included inside the Logistic Regression and SVM pipelines so it is fitted only on training data. Random Forest does not require feature scaling.

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=RANDOM_STATE, stratify=y
)
print("Training rows:", len(X_train))
print("Testing rows:", len(X_test))

## 3. Model implementation

Three classification algorithms are used: Logistic Regression, Random Forest, and Support Vector Machine.

In [ ]:
models = {
    "Logistic Regression": Pipeline([
        ("scaler", StandardScaler()),
        ("classifier", LogisticRegression(max_iter=5000, random_state=RANDOM_STATE))
    ]),
    "Random Forest": RandomForestClassifier(
        n_estimators=300, random_state=RANDOM_STATE, class_weight="balanced"
    ),
    "Support Vector Machine": Pipeline([
        ("scaler", StandardScaler()),
        ("classifier", SVC(probability=True, random_state=RANDOM_STATE))
    ])
}

## 4. Evaluation

Accuracy, precision, recall, F1-score, and ROC-AUC are calculated on the held-out test set. In this dataset, class `1` is `benign`; the metrics below therefore use benign as the positive class.

In [ ]:
records = []
for name, model in models.items():
    model.fit(X_train, y_train)
    predictions = model.predict(X_test)
    probabilities = model.predict_proba(X_test)[:, 1]
    records.append({
        "Model": name,
        "Accuracy": accuracy_score(y_test, predictions),
        "Precision": precision_score(y_test, predictions, zero_division=0),
        "Recall": recall_score(y_test, predictions, zero_division=0),
        "F1-score": f1_score(y_test, predictions, zero_division=0),
        "ROC-AUC": roc_auc_score(y_test, probabilities)
    })

metrics = pd.DataFrame(records).sort_values("F1-score", ascending=False)
display(metrics.round(4))
metrics.to_csv(RESULTS_DIR / "model_metrics.csv", index=False)

## 5. Confusion matrices

In [ ]:
for name, model in models.items():
    fig, ax = plt.subplots(figsize=(5, 4))
    ConfusionMatrixDisplay.from_estimator(
        model, X_test, y_test,
        display_labels=dataset.target_names, cmap="Blues", ax=ax
    )
    ax.set_title(f"Confusion Matrix — {name}")
    fig.tight_layout()
    filename = name.lower().replace(" ", "_")
    fig.savefig(RESULTS_DIR / f"confusion_matrix_{filename}.png", dpi=160)
    plt.show()
    plt.close(fig)

## 6. ROC curves and model comparison

In [ ]:
fig, ax = plt.subplots(figsize=(7, 5))
for name, model in models.items():
    RocCurveDisplay.from_estimator(model, X_test, y_test, name=name, ax=ax)
ax.set_title("ROC Curves — Test Set")
fig.tight_layout()
fig.savefig(RESULTS_DIR / "roc_curves.png", dpi=160)
plt.show()
plt.close(fig)

chart = metrics.set_index("Model")[["Accuracy", "Precision", "Recall", "F1-score", "ROC-AUC"]]
ax = chart.plot(kind="bar", figsize=(10, 6), ylim=(0, 1.05))
ax.set_title("Model Performance Comparison")
ax.set_ylabel("Score")
ax.set_xlabel("")
plt.xticks(rotation=15, ha="right")
plt.tight_layout()
plt.savefig(RESULTS_DIR / "model_comparison.png", dpi=160)
plt.show()
plt.close()

## 7. Findings and conclusion

Compare the models using the displayed test-set metrics. The model with the highest score is not automatically suitable for clinical use. Consider the consequences of false negatives and false positives, class definitions, data representativeness, external validation, and clinical review before drawing real-world conclusions.

**Reproducibility:** the train/test split and model random states use seed `42`. Results may vary if the data, preprocessing, or parameters are changed.